In [ ]:
import rootutils
from pathlib import Path
PROJECT_ROOT = rootutils.setup_root(
    search_from=Path.cwd(), dotenv=True, pythonpath=True, cwd=False)

import numpy as np
# from numpy.typing import NDArray
# from typing import Tuple

from src.zupt_ins.zupt_ins import smoothed_zupt_aided_ins
from src.zupt_ins.data_classes import InertialData, Trajectory
from src.zupt_ins.initialization import INSConfig

### Load data

In [ ]:
inertial = InertialData.from_csv_int(PROJECT_ROOT / "data/angermann_high_precision", 15)
gt_traj = Trajectory.from_csv_int(PROJECT_ROOT / "data/angermann_high_precision", 15)
simdata = INSConfig(
    segmentation_thrsld=0.03 # segments the steps
)

### Process data with Smoothed ZUPT aided INS

In [ ]:
zupt, ins_traj, segs = smoothed_zupt_aided_ins(inertial, simdata)

### Plot

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
from src.plotting.plot_trajectories import plot_groundtruth_vs_inertial_positions

plot_groundtruth_vs_inertial_positions(ins_traj, gt_traj)


### Test using Ground thruth timestamps aligned to IMU measurements

In [ ]:

gt_traj_aligned = gt_traj.temporal_alignment(ins_traj.t)

fig, ax = plt.subplots()
ax.plot(ins_traj.pos[0, :-1], ins_traj.pos[1, :-1])
ax.set_aspect('equal')
ax.grid(visible=True)

In [ ]:
## Compare Groundtruths
fig, ax = plt.subplots()
ax.plot(gt_traj.pos[0,:-1], gt_traj.pos[1,:-1])
ax.plot(gt_traj_aligned.pos[0,:-1], gt_traj_aligned.pos[1,:-1], linewidth = 0.5)
ax.grid(visible=True)
plt.show()

### Plot Segmentation and inertial power

In [ ]:
fig, ax = plt.subplots(figsize=(10,5))
ax.plot(inertial.t, (inertial.u[1:3,:]**2).sum(axis=0), linewidth=0.2, zorder=1)
ax.scatter(inertial.t[segs], 100*np.ones_like(segs), marker='x', c='r', s=0.3, zorder=2)
ax.grid(visible=True)
plt.show()